## Frecuencia respiratoria desde PPG de dedo en sujetos sanos despiertos (CASE) — estudio final
**Pregunta:** ¿qué método estima mejor la RR desde la PPG en sujetos sanos y despiertos, y sirve esa RR
para detectar activación emocional como la de una banda torácica?

**Protocolo (fijado antes de ver los resultados de prueba):**
- Ventanas de 30 s (paso 10 s; análisis secundario con 60 s cambiando `VENTANA` en c-1) dentro de un vídeo emocional, sin artefacto en la PPG y con referencia de banda
  fiable (conteo de respiraciones y FFT de la banda coinciden ≤ 2 rpm).
- División por participantes de los notebooks 09 y 12: **15 de desarrollo / 15 de prueba** (semilla 2026).
- Se **elige** el método con desarrollo (menor MAE mediano por participante entre los que estiman ≥ 80 % de las
  ventanas) y se **informa** en prueba. Todos los métodos de reglas tienen parámetros fijados de antemano.
- Comparaciones pareadas por participante (Wilcoxon) en prueba, sobre las ventanas que ambos métodos estiman.
- Utilidad: clasificación de activación (vídeos divertido/miedo frente a aburrido/relajante) dejando un
  participante fuera, con HRV de la PPG ± RR de la banda o de la PPG.

**Métodos:**
| | Método | Tipo |
|---|---|---|
| A | FFT de la RIIV, máximo global 6–35 rpm (config congelada BIDMC/CapnoBase) | espectral estándar |
| SF | Tipo Smart Fusion (Karlen et al., 2013): FFT de RIIV, RIAV, RIFV; media si DE ≤ 4 rpm, si no abstención | espectral estándar |
| B | Mediana de los máximos de las 3 modulaciones, banda ≥ 10 rpm (notebook 09) | espectral con banda ajustada |
| S | FFT con corrección del fondo 1/f: pico de mayor prominencia sobre el fondo ajustado | espectral propuesto |
| S-AR | Igual que S con espectro autorregresivo (Burg, orden 10) | espectral propuesto |
| D | Conteo de respiraciones en el tiempo, pasa-alto 0.17 Hz | temporal |
| Da | Conteo con pasa-alto adaptativo 0.6 × RR(S) | temporal propuesto |
| F | Fusión S + Da: media si coinciden ± 2 rpm, si no abstención | fusión propuesta |
| F-t | F con fusión temporal: mediana de las últimas 3 estimaciones disponibles del mismo vídeo | fusión propuesta |
| CNN | CNN-LSTM sobre PPG cruda (notebook 12; validación cruzada en desarrollo, entrenada con desarrollo) | aprendizaje profundo |

In [ ]:
# c-1 — Imports, configuración y rutas
import json
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import butter, decimate, filtfilt, find_peaks, resample_poly
from scipy.stats import wilcoxon
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE_DIR / "notebooks"))
sys.path.insert(0, str(Path.cwd()))
import rr_generalizable as rg

VENTANA = 30                                  # s. Análisis principal: 30. Análisis secundario: cambia a 60.
rg.VENTANA_S, rg.N = VENTANA, VENTANA * rg.FS  # el módulo usa estos valores en cada ventana

OUT = BASE_DIR / "results"
RES = OUT if VENTANA == 30 else OUT / f"ventana_{VENTANA}s"   # los resultados de 60 s no pisan los de 30 s
RES.mkdir(exist_ok=True)
CFG = json.loads((OUT / "config_congelada.json").read_text())
SPLIT = json.loads((OUT / "config_case_correccion.json").read_text())
CASE_DEV = set(SPLIT["case_desarrollo"])
P = {**rg.PARAMS, "rpm_min": CFG["rpm_min"], "rpm_max": CFG["rpm_max"], "lp_riiv_hz": CFG["lp_fft_hz"],
     "lp_art_hz": CFG["lp_art_hz"], "umbral_art": CFG["umbral_art"]}
VENTANA_S, PASO_S, FS = rg.VENTANA_S, rg.PASO_S, rg.FS
CASE_DIR = BASE_DIR / "data" / "CASE" / "data" / "interpolated" / "physiological"
ARCHIVOS = sorted(CASE_DIR.glob("sub_*.csv"), key=lambda p: int(p.stem.split("_")[1]))
ACTIVACION = {1: 1, 2: 1, 7: 1, 8: 1, 3: 0, 4: 0, 5: 0, 6: 0}
COB_MIN = 80                                  # % mínimo de ventanas estimadas para poder ser elegido
pd.set_option("display.width", 250)
warnings.filterwarnings("ignore", category=RuntimeWarning)   # correlaciones de sujetos sin variación
print(f"{len(ARCHIVOS)} participantes | desarrollo: {len(CASE_DEV)} | ventana {VENTANA} s, paso {PASO_S} s | resultados en {RES}")

In [ ]:
# c-2 — Funciones auxiliares (referencia, métodos espectrales estándar, HRV)
def rr_ref(m, t0, t1, min_marcas=3):
    m = m[(m >= t0) & (m < t1)]
    return np.nan if len(m) < min_marcas else 60.0 / np.median(np.diff(m))

def marcas_banda(rsp, fs=1000):
    x = decimate(decimate(rsp, 10, zero_phase=True), 4, zero_phase=True)
    f = fs / 40
    b, a = butter(2, [0.05, 1.0], btype="band", fs=f)
    x = filtfilt(b, a, x)
    iqr = np.subtract(*np.percentile(x, [75, 25]))
    pk, _ = find_peaks(x, distance=int(f), prominence=0.25 * iqr)
    return pk / f

def maximo(x, lo, hi=P["rpm_max"]):
    f, Pw = rg.espectro(x, rg.FS_M, 1024)
    b = (f >= lo) & (f <= hi)
    return f[b][np.argmax(Pw[b])]

def hrv(t, ibi, t0):
    m = (t >= t0) & (t < t0 + VENTANA_S)
    if m.sum() < 15:
        return np.nan, np.nan, np.nan
    tw, iw = t[m], ibi[m]
    consec = np.abs(np.diff(tw) - iw[1:]) < 0.02
    d = np.diff(iw)[consec]
    rmssd = np.log(1000 * np.sqrt(np.mean(d ** 2))) if consec.sum() >= 10 else np.nan
    return 60 / iw.mean(), rmssd, np.log(1000 * iw.std())

In [ ]:
# c-3 — Ventanas y estimaciones de todos los métodos (tarda ~10–15 min)
filas, t_ini = [], time.time()
for p in ARCHIVOS:
    d = pd.read_csv(p, usecols=["bvp", "rsp", "video"])
    vid = d["video"].to_numpy(int)
    rsp = d["rsp"].to_numpy(float)
    marcas = marcas_banda(rsp)
    rsp5 = decimate(decimate(resample_poly(rsp, FS, 1000), 5, zero_phase=True), 5, zero_phase=True)
    est = rg.Estimador(d["bvp"].to_numpy(float), 1000, P)
    for t0 in est.inicios():
        v = vid[int(t0 * 1000):int((t0 + VENTANA_S) * 1000)]
        if len(v) == 0 or v.min() != v.max() or v[0] not in ACTIVACION:
            continue
        r = rr_ref(marcas, t0, t0 + VENTANA_S)
        seg = rsp5[int(t0 * 5):int((t0 + VENTANA_S) * 5)]
        if not np.isfinite(r) or not P["rpm_min"] <= r <= P["rpm_max"] or len(seg) < VENTANA_S * 5:
            continue
        f, Pw = rg.espectro(seg, 5, 2048)
        bnd = (f >= P["rpm_min"]) & (f <= P["rpm_max"])
        if abs(r - f[bnd][np.argmax(Pw[bnd])]) > 2:
            continue                                      # referencia no fiable
        w = est.ventana(t0)
        series = [rg.serie_4hz(est.t, s, t0) for s in est.mods.values()]
        ok = [s for s in series if s is not None]
        sf = [maximo(s, P["rpm_min"]) for s in ok]
        fc, rmssd, sdnn = hrv(est.t, est.mods["RIFV"], t0)
        filas.append({"Participante": p.stem, "Conjunto": "desarrollo" if p.stem in CASE_DEV else "prueba",
                      "Inicio (s)": t0, "Vídeo": int(v[0]), "Activación": ACTIVACION[int(v[0])],
                      "RR banda": r, "Artefacto": w["Artefacto"],
                      "A": w["A"],
                      "SF": np.mean(sf) if len(sf) == 3 and np.std(sf) <= 4 else np.nan,
                      "B": rg.mediana3([maximo(s, 10) for s in ok] + [np.nan] * (3 - len(ok))),
                      "S": w["S"], "S-AR": w["S-AR"], "D": w["D"], "Da": w["Da"], "F": w["F"],
                      "FC": fc, "lnRMSSD": rmssd, "lnSDNN": sdnn})
    print(p.stem, end=" ")
R = pd.DataFrame(filas).sort_values(["Participante", "Inicio (s)"]).reset_index(drop=True)

# Fusión temporal: mediana de las últimas 3 estimaciones F disponibles dentro del mismo vídeo (causal)
bloque = ((R["Participante"] != R["Participante"].shift()) | (R["Vídeo"] != R["Vídeo"].shift())).cumsum()
R["F-t"] = R.groupby(bloque)["F"].transform(lambda s: s.rolling(3, min_periods=1).median())
R.loc[R["F"].isna(), "F-t"] = np.nan                       # se mantiene la abstención de la ventana actual

# CNN-LSTM del notebook 12 (si existe su archivo de predicciones; solo con ventanas de 30 s, como se entrenó)
f12 = OUT / "case_rr_mejor_ventanas.csv"
if VENTANA == 30 and f12.exists() and "G: CNN-LSTM (PPG cruda)" in pd.read_csv(f12, nrows=0).columns:
    c12 = pd.read_csv(f12, usecols=["Participante", "Inicio (s)", "G: CNN-LSTM (PPG cruda)"])
    R = R.merge(c12.rename(columns={"G: CNN-LSTM (PPG cruda)": "CNN"}), on=["Participante", "Inicio (s)"], how="left")
METODOS = ["A", "SF", "B", "S", "S-AR", "D", "Da", "F", "F-t"] + (["CNN"] if "CNN" in R else [])
R.to_csv(RES / "case_final_ventanas.csv", index=False)
V = R[~R["Artefacto"]].copy()
print(f"\n{len(R)} ventanas con referencia fiable; {len(V)} sin artefacto ({time.time() - t_ini:.0f} s)")
print(V.groupby("Conjunto")["Participante"].agg(["nunique", "size"]))

In [ ]:
# c-4 — Métricas
def metricas(g, col):
    ok = g[col].notna()
    if ok.sum() == 0:
        return {"Cobertura %": 0.0}
    h = g[ok]
    err = h[col] - h["RR banda"]
    e = err.abs()
    r = h.groupby("Participante").apply(lambda s: s[col].corr(s["RR banda"]) if s[col].std() > 0 else np.nan)
    dl = h.groupby(["Participante", "Activación"])[[col, "RR banda"]].mean().unstack()
    if (col, 1) in dl and (col, 0) in dl:
        dm, dr = dl[(col, 1)] - dl[(col, 0)], dl[("RR banda", 1)] - dl[("RR banda", 0)]
    else:
        dm = dr = pd.Series(dtype=float)
    return {"MAE mediano": e.groupby(h["Participante"]).mean().median(), "MAE medio": e.mean(),
            "RMSE": np.sqrt((err ** 2).mean()), "Sesgo": err.mean(),
            "LdA inf": err.mean() - 1.96 * err.std(), "LdA sup": err.mean() + 1.96 * err.std(),
            "% ≤ 2 rpm": 100 * (e <= 2).mean(), "Cobertura %": 100 * ok.mean(),
            "r dentro de sujeto": r.median(), "Δ método": dm.median(), "Δ banda": dr.median(),
            "r de los Δ": dm.corr(dr)}

TAB = {c: pd.DataFrame({m: metricas(V[V["Conjunto"] == c] if c != "todos" else V, m) for m in METODOS}).T
       for c in ["desarrollo", "prueba", "todos"]}
pd.concat(TAB, names=["Conjunto", "Método"]).to_csv(RES / "case_final_resumen.csv")
print("=== DESARROLLO (para elegir) ===")
print(TAB["desarrollo"].sort_values("MAE mediano").round(2).to_string())
cand = TAB["desarrollo"][(TAB["desarrollo"]["Cobertura %"] >= COB_MIN) & TAB["desarrollo"]["MAE mediano"].notna()]
ELEGIDO = cand["MAE mediano"].idxmin()
print(f"\nMétodo elegido en desarrollo: {ELEGIDO}")

In [ ]:
# c-5 — PRUEBA: resultados y comparaciones pareadas con el método elegido
print("=== PRUEBA (evaluación honesta) ===")
print(TAB["prueba"].sort_values("MAE mediano").round(2).to_string())
T = V[V["Conjunto"] == "prueba"]
filas = []
for m in METODOS:
    if m == ELEGIDO:
        continue
    comun = T[T[ELEGIDO].notna() & T[m].notna()]
    ea = (comun[ELEGIDO] - comun["RR banda"]).abs().groupby(comun["Participante"]).mean()
    eb = (comun[m] - comun["RR banda"]).abs().groupby(comun["Participante"]).mean()
    dlt = (ea - eb).dropna()
    filas.append({"Comparación": f"{ELEGIDO} vs {m}", "Participantes": len(dlt), "Ventanas comunes": len(comun),
                  f"MAE {ELEGIDO}": ea.median(), f"MAE otro": eb.median(), "Δ mediana": dlt.median(),
                  "Wilcoxon p": wilcoxon(dlt).pvalue if (dlt != 0).any() and len(dlt) > 1 else np.nan})
CMP = pd.DataFrame(filas)
CMP.to_csv(RES / "case_final_comparaciones.csv", index=False)
print("\nComparaciones pareadas por participante (prueba, ventanas comunes; Δ < 0 = el elegido es mejor):")
print(CMP.round(3).to_string(index=False))

In [ ]:
# c-6 — Compromiso exactitud–cobertura de la fusión (todos los participantes)
filas = []
for tol in [0.5, 1, 1.5, 2, 3, 4, 6, np.inf]:
    col = f"F tol {tol}"
    V[col] = rg.fusion(V["S"], V["Da"], tol) if np.isfinite(tol) else (V["S"] + V["Da"]) / 2
    m = metricas(V, col)
    filas.append({"Tolerancia (rpm)": tol, "MAE mediano": m.get("MAE mediano"), "% ≤ 2 rpm": m.get("% ≤ 2 rpm"),
                  "Cobertura %": m["Cobertura %"], "r dentro de sujeto": m.get("r dentro de sujeto")})
CC = pd.DataFrame(filas)
print(CC.round(2).to_string(index=False))
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(CC["Cobertura %"], CC["MAE mediano"], "o-")
for _, r_ in CC.iterrows():
    ax.annotate(f"±{r_['Tolerancia (rpm)']:g}", (r_["Cobertura %"], r_["MAE mediano"]), fontsize=8,
                xytext=(4, 4), textcoords="offset points")
for m, mk in [("A", "s"), ("S", "^"), ("D", "v")]:
    mm = metricas(V, m)
    ax.plot(mm["Cobertura %"], mm["MAE mediano"], mk, label=m)
ax.set_xlabel("Cobertura (% de ventanas estimadas)")
ax.set_ylabel("MAE mediano por participante (rpm)")
ax.set_yscale("log")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(RES / "case_final_cobertura.png", dpi=300)
plt.show()

In [ ]:
# c-7 — Figuras de acuerdo en prueba: FFT estándar frente al método elegido
fig, axs = plt.subplots(2, 2, figsize=(9, 8))
for j, (nombre, col) in enumerate([("A: FFT estándar", "A"), (f"{ELEGIDO} (elegido)", ELEGIDO)]):
    h = T[T[col].notna()]
    axs[0, j].scatter(h["RR banda"], h[col], s=4, alpha=0.3)
    axs[0, j].plot([6, 35], [6, 35], "k--", lw=0.8)
    axs[0, j].set_xlabel("RR banda (rpm)")
    axs[0, j].set_ylabel("RR PPG (rpm)")
    axs[0, j].set_title(nombre, fontsize=10)
    media, dif = (h[col] + h["RR banda"]) / 2, h[col] - h["RR banda"]
    axs[1, j].scatter(media, dif, s=4, alpha=0.3)
    for y_, ls in [(dif.mean(), "-"), (dif.mean() + 1.96 * dif.std(), "--"), (dif.mean() - 1.96 * dif.std(), "--")]:
        axs[1, j].axhline(y_, c="k", ls=ls, lw=0.8)
    axs[1, j].set_xlabel("Media (rpm)")
    axs[1, j].set_ylabel("PPG − banda (rpm)")
    axs[1, j].set_title(f"sesgo {dif.mean():+.2f}, LdA ±{1.96 * dif.std():.2f} rpm", fontsize=9)
plt.tight_layout()
plt.savefig(RES / "case_final_acuerdo.png", dpi=300)
plt.show()

In [ ]:
# c-8 — Utilidad: detección de activación (dejando un participante fuera, todos los participantes)
# Todos los brazos usan las mismas ventanas: aquellas donde el método elegido estima (si se abstiene, la ventana
# se excluye para todos), para que la comparación sea justa.
BRAZOS = {"HRV": ["FC", "lnRMSSD", "lnSDNN"],
          "HRV + RR banda": ["FC", "lnRMSSD", "lnSDNN", "RR banda"],
          f"HRV + RR PPG ({ELEGIDO})": ["FC", "lnRMSSD", "lnSDNN", ELEGIDO],
          "HRV + RR PPG (A, FFT estándar)": ["FC", "lnRMSSD", "lnSDNN", "A"],
          "RR banda sola": ["RR banda"],
          f"RR PPG sola ({ELEGIDO})": [ELEGIDO]}
cols = sorted({c for cs in BRAZOS.values() for c in cs})
C = V.dropna(subset=cols).copy()
C[cols] = C.groupby("Participante")[cols].transform(lambda c: (c - c.mean()) / (c.std() + 1e-9))
y, part = C["Activación"].to_numpy(), C["Participante"].to_numpy()
print(f"Ventanas para clasificar: {len(C)} ({C['Participante'].nunique()} participantes)")
AUC = []
for brazo, cs in BRAZOS.items():
    X = C[cs].to_numpy()
    for s in np.unique(part):
        te = part == s
        if len(np.unique(y[te])) < 2:
            continue
        clf = LogisticRegression(max_iter=1000).fit(X[~te], y[~te])
        AUC.append({"Brazo": brazo, "Participante": s, "AUC": roc_auc_score(y[te], clf.predict_proba(X[te])[:, 1])})
A_ = pd.DataFrame(AUC)
A_.to_csv(RES / "case_final_auc.csv", index=False)
print(A_.groupby("Brazo", sort=False)["AUC"].agg(["count", "median", "mean"]).round(3).to_string())
W = A_.pivot(index="Participante", columns="Brazo", values="AUC")
for a, b in [("HRV + RR banda", "HRV"), (f"HRV + RR PPG ({ELEGIDO})", "HRV"),
             (f"HRV + RR PPG ({ELEGIDO})", "HRV + RR banda"), (f"HRV + RR PPG ({ELEGIDO})", "HRV + RR PPG (A, FFT estándar)")]:
    dlt = (W[a] - W[b]).dropna()
    p_ = wilcoxon(dlt).pvalue if (dlt != 0).any() and len(dlt) > 1 else np.nan
    print(f"{a}  vs  {b}: Δ AUC mediana {dlt.median():+.3f}, Wilcoxon p = {p_:.3f}")

In [ ]:
# c-9 — Configuración final para el wearable
(RES / "config_case_final.json").write_text(json.dumps(
    {"metodo_elegido": ELEGIDO, "params": {**P, "fondo_rpm": list(P["fondo_rpm"])},
     "criterio": f"menor MAE mediano en desarrollo con cobertura >= {COB_MIN} %",
     "ventana_s": VENTANA, "paso_s": PASO_S, "case_desarrollo": sorted(CASE_DEV)}, indent=2, default=float))
print(f"Guardado: {RES / 'config_case_final.json'}")